## Defining architechure and train function


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

transform_train = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

transform_test = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.ImageFolder('content/fer2013plus/fer2013plus/fer2013/train/', transform=transform_train)
test_dataset  = datasets.ImageFolder('content/fer2013plus/fer2013plus/fer2013/test/', transform=transform_test)

BATCH_SIZE = 512 

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    num_workers=4,           
    pin_memory=True,         
    persistent_workers=True, 
    drop_last=True           
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

class FER_DCNN(nn.Module):
    def __init__(self):
        super(FER_DCNN, self).__init__()

        def conv_block(in_f, out_f, dropout_rate):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ELU(inplace=True),
                nn.Conv2d(out_f, out_f, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ELU(inplace=True),
                nn.MaxPool2d(2),
                nn.Dropout(dropout_rate)
            )

        # 1. Conv block count: 3 sets
        # 2. Filter counts: 64, 128, 256
        # 3. Dropout (conv): Rate 0.3 for all blocks
        self.features = nn.Sequential(
            conv_block(1, 64, 0.3),   # Block 1
            conv_block(64, 128, 0.3),  # Block 2
            conv_block(128, 256, 0.3)  # Block 3
        )

        # 4. Dense layers: 1 dense, 128 neurons
        # Spatial size logic: 48 -> 24 -> 12 -> 6 (after 3 MaxPools)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 128),
            nn.BatchNorm1d(128),
            nn.ELU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(128, 8)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FER_DCNN().to(device=device, memory_format=torch.channels_last)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=0.001, fused=True)  # Lower initial LR

# Warmup for 5 epochs manually, then cosine decay
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

scaler = torch.amp.GradScaler('cuda')

def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0

    with torch.no_grad(), torch.amp.autocast('cuda'):
        for images, labels in loader:
            images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels) # Calculate loss
            _, preds = torch.max(outputs, 1)

            total_loss += loss.item() # Accumulate loss
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total, total_loss / len(loader)

## Training Model

In [2]:
# Training loop
best_acc, best_val_loss = 0, float('inf')
patience, no_improve_count, epoch = 15, 0, 0

while True:
    if epoch < 5:
        for param_group in optimizer.param_groups:
            param_group['lr'] = 1e-6 + (0.001 - 1e-6) * ((epoch + 1) / 5)

    train_loss = train_epoch(model, train_loader)
    val_acc, val_loss = evaluate(model, test_loader)

    if epoch >= 5:
        scheduler.step()

    if val_acc > best_acc + 1e-4:
        best_acc = val_acc
        no_improve_count = 0
        torch.save(model.state_dict(), "best_fer_model.pth")
    else:
        no_improve_count += 1

    print(f"Epoch {epoch+1}: TrainLoss={train_loss:.4f}  ValLoss={val_loss:.4f}  "
          f"ValAcc={val_acc:.4f}  NoImprove={no_improve_count}/{patience}")

    if no_improve_count >= patience:
        print(f"Converged at epoch {epoch+1}. Best Val Acc: {best_acc:.4f}")
        break

    epoch += 1

Epoch 1: TrainLoss=1.6661  ValLoss=1.3652  ValAcc=0.5566  NoImprove=0/15
Epoch 2: TrainLoss=1.2264  ValLoss=1.0804  ValAcc=0.6359  NoImprove=0/15
Epoch 3: TrainLoss=1.0432  ValLoss=0.9432  ValAcc=0.6694  NoImprove=0/15
Epoch 4: TrainLoss=0.9352  ValLoss=0.8975  ValAcc=0.6733  NoImprove=0/15
Epoch 5: TrainLoss=0.8732  ValLoss=0.8060  ValAcc=0.7177  NoImprove=0/15
Epoch 6: TrainLoss=0.8079  ValLoss=0.7778  ValAcc=0.7281  NoImprove=0/15
Epoch 7: TrainLoss=0.7711  ValLoss=0.7316  ValAcc=0.7394  NoImprove=0/15
Epoch 8: TrainLoss=0.7372  ValLoss=0.7283  ValAcc=0.7439  NoImprove=0/15
Epoch 9: TrainLoss=0.7111  ValLoss=0.7233  ValAcc=0.7463  NoImprove=0/15
Epoch 10: TrainLoss=0.6875  ValLoss=0.6932  ValAcc=0.7524  NoImprove=0/15
Epoch 11: TrainLoss=0.6678  ValLoss=0.6827  ValAcc=0.7584  NoImprove=0/15
Epoch 12: TrainLoss=0.6497  ValLoss=0.6629  ValAcc=0.7629  NoImprove=0/15
Epoch 13: TrainLoss=0.6295  ValLoss=0.6652  ValAcc=0.7642  NoImprove=0/15
Epoch 14: TrainLoss=0.6125  ValLoss=0.6630  Val